# 고독사·독거노인 뉴스 텍스트마이닝 — 최종 보고서

> **구글 코랩에서 그대로 실행되는 자기완결형 노트북.**
> `src` import 없이 모든 함수를 셀에 직접 정의한다. 위에서 아래로 실행하면 전체 분석이 재현된다.

**분석 대상**: 빅카인즈 뉴스 (2025-06-01 ~ 2026-06-01, 키워드 OR 검색 전수)
**파이프라인**: 로드·정제 → 빈도·워드클라우드 → LDA 토픽 → 동시출현 네트워크 → 감성 분석

---

## 핵심 결론 (요약)

**고독사·독거노인은 사회적 '위험' 이슈지만, 언론 보도는 주로 복지·나눔이라는 '시혜적·긍정적' 프레임으로 소비된다.**
이 모순이 데이터 전반에서 일관되게 나타난다.

- 담론은 6개 주제로 갈리며, **시혜·복지 프레임(약 50%)이 위기 프레임(약 38%)보다 우세**.
- 전체 감성은 **긍정 약 61% / 부정 약 23%** 로 긍정 편향.
- 단, **죽음·재난을 다루는 주제(고독사예방·재난안전대응)에서만 부정 비율이 높다(약 27~30%)** — "위험을 말할 때만 부정적"인 구조.
- 핵심어 네트워크의 중심은 `복지·가구·독거노인·취약·계층` — 모든 담론이 **'복지 대상으로서의 독거노인'**에 수렴.

## 0. 실행 환경 준비 (Colab)

아래 4개 셀을 순서대로 실행한다: ① 패키지 설치 ② 한글 폰트 설치 ③ 드라이브 마운트 ④ 감성사전 다운로드.
로컬에서 돌릴 경우 ①~③은 건너뛰고, 경로만 맞추면 된다.

In [ ]:
# ① 패키지 설치 (Colab)
!pip install -q konlpy wordcloud pyLDAvis gensim networkx openpyxl
# konlpy(Okt)는 JVM이 필요하다. Colab에는 Java가 기본 설치되어 있다.

In [ ]:
# ② 한글 폰트 설치 (나눔고딕)
!apt-get -qq install -y fonts-nanum > /dev/null

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(NANUM)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
print("폰트 등록:", plt.rcParams["font.family"])

In [ ]:
# ③ 구글 드라이브 마운트 + 프로젝트 경로 설정
from google.colab import drive
drive.mount("/content/drive")

# ↓↓↓ 본인 드라이브에 맞게 수정 (빅카인즈 엑셀들을 RAW_DIR에 넣어둔다) ↓↓↓
PROJECT_DIR = "/content/drive/MyDrive/BDA"
RAW_DIR = f"{PROJECT_DIR}/data/raw"

import glob
print("발견된 엑셀:", glob.glob(f"{RAW_DIR}/*.xlsx"))

In [ ]:
# ④ KNU 한국어 감성사전 다운로드 (감성 분석용)
!wget -q -O /content/SentiWord_info.json https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/data/SentiWord_info.json
LEXICON_PATH = "/content/SentiWord_info.json"
import os; print("감성사전 크기(bytes):", os.path.getsize(LEXICON_PATH))

## 0-1. 설정 — 키워드 & 불용어

In [ ]:
# 수집 키워드 (참고용)
KEYWORDS = ["독거노인", "고독사", "1인 고령가구", "노인 고립", "무연고 사망"]

# 변별력 없는 행정·보도자료 상투어 (분석에서 제거). 핵심어(복지/가구/취약 등)는 남긴다.
STOP_WORDS = set([
    "기자", "뉴스", "사진", "제공", "무단", "전재", "재배포", "금지", "연합뉴스",
    "오전", "오후", "이날", "지난", "올해", "최근", "관련", "통해", "위해", "대한",
    "이번", "당시", "이상", "이후", "이전", "정도", "가운데", "경우", "때문",
    "지역", "사회", "사업", "지원", "추진", "협의", "전달", "나눔", "서비스",
    "운영", "마련", "실시", "진행", "활동", "대상", "계획", "참여", "행사",
])

## 0-2. 함수 정의 (전처리 / 토픽 / 네트워크 / 감성)

`src/*.py`의 로직을 코랩에서 자기완결로 돌리기 위해 셀에 직접 풀어쓴다.

In [ ]:
import re, json
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
from konlpy.tag import Okt

okt = Okt()

# ---- 전처리 ----
_URL = re.compile(r"https?://\S+|www\.\S+")
_EMAIL = re.compile(r"\S+@\S+")
_NONKO = re.compile(r"[^가-힣\s]")
_SP = re.compile(r"\s+")

def clean_text(t):
    if not isinstance(t, str): return ""
    t = _URL.sub(" ", t); t = _EMAIL.sub(" ", t)
    t = _NONKO.sub(" ", t); t = _SP.sub(" ", t)
    return t.strip()

# 복합어(독거노인 등) 보존: Okt가 쪼갠 조각을 다시 합친다
_COMP = [w for w in KEYWORDS]
_COMP_PARTS = sorted([(w, tuple(okt.nouns(w))) for w in _COMP],
                     key=lambda x: -len(x[1]))

def _merge_compounds(nouns):
    res, i, n = [], 0, len(nouns)
    while i < n:
        hit = False
        for w, parts in _COMP_PARTS:
            k = len(parts)
            if k >= 2 and tuple(nouns[i:i+k]) == parts:
                res.append(w); i += k; hit = True; break
        if not hit:
            res.append(nouns[i]); i += 1
    return res

def extract_nouns(text, min_len=2):
    c = clean_text(text)
    if not c: return []
    nouns = _merge_compounds(okt.nouns(c))
    return [w for w in nouns if len(w) >= min_len and w not in STOP_WORDS]

# ---- 감성 토큰 (형용사/동사 어간까지) ----
_SENTI_POS = {"Noun", "Adjective", "Verb", "Adverb"}
def senti_tokens(text, min_len=2):
    c = clean_text(text)
    if not c: return []
    return [w for w, tag in okt.pos(c, norm=True, stem=True)
            if tag in _SENTI_POS and len(w) >= min_len]


In [ ]:
# ---- 감성사전 ----
def load_lexicon(path):
    data = json.load(open(path, encoding="utf-8"))
    lex = {}
    for x in data:
        w = x["word"]
        if " " in w or not re.fullmatch(r"[가-힣]+", w): continue
        lex[w] = int(x["polarity"])
    return lex

def score_document(text, lexicon):
    score = pos = neg = n = 0
    for w in senti_tokens(text):
        p = lexicon.get(w)
        if p is None: continue
        score += p; n += 1
        if p > 0: pos += 1
        elif p < 0: neg += 1
    label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return score, pos, neg, n, label

# ---- 네트워크 ----
def top_vocab(token_lists, top_n):
    cnt = Counter(w for t in token_lists for w in t)
    return [w for w, _ in cnt.most_common(top_n)]

def build_network(token_lists, top_n=50, min_cooc=300):
    import networkx as nx
    vocab = top_vocab(token_lists, top_n); vset = set(vocab)
    freq = Counter(w for t in token_lists for w in t)
    pairs = Counter()
    for toks in token_lists:
        present = sorted(vset.intersection(toks))
        for a, b in combinations(present, 2): pairs[(a, b)] += 1
    G = nx.Graph()
    for w in vocab: G.add_node(w, freq=int(freq[w]))
    for (a, b), c in pairs.items():
        if c >= min_cooc: G.add_edge(a, b, weight=int(c))
    G.remove_nodes_from([n for n in list(G.nodes) if G.degree(n) == 0])
    return G

def centrality(G):
    import networkx as nx
    deg = nx.degree_centrality(G); bet = nx.betweenness_centrality(G, weight="weight")
    rows = [{"node": n, "degree": round(deg[n], 4),
             "betweenness": round(bet[n], 4), "freq": G.nodes[n]["freq"]} for n in G.nodes]
    return pd.DataFrame(rows).sort_values("degree", ascending=False).reset_index(drop=True)


## 1. 데이터 로드 & 정제

`data/raw/`의 엑셀을 모두 읽어 병합한다. **`뉴스 식별자`는 26자리 문자열이므로 반드시 `dtype=str`로 읽는다**
(숫자로 읽으면 부동소수점 정밀도 손실로 중복제거가 깨진다).

In [ ]:
ID, DATE, TITLE, BODY, EXC = "뉴스 식별자", "일자", "제목", "본문", "분석제외 여부"
EXCLUDE_FLAGS = {"예외", "중복", "중복, 예외"}

files = sorted(glob.glob(f"{RAW_DIR}/*.xlsx"))
df = pd.concat([pd.read_excel(f, dtype={ID: str}) for f in files], ignore_index=True)
df = df.drop_duplicates(subset=ID).reset_index(drop=True)
print("병합(식별자 중복제거):", df.shape)

if EXC in df.columns:
    df = df[~df[EXC].isin(EXCLUDE_FLAGS)]
df = df.drop_duplicates(subset=[TITLE, BODY])
df["날짜"] = pd.to_datetime(df[DATE].astype(str), format="%Y%m%d", errors="coerce")
df = df.reset_index(drop=True)
print("정제 후:", df.shape, "| 기간:", df["날짜"].min().date(), "~", df["날짜"].max().date())

## 2. 빈도 분석 & 워드클라우드

본문에서 Okt로 명사를 추출한다(약 1~2분). 결과는 `df["nouns"]`에 저장해 이후 단계에서 재사용한다.

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Okt 명사추출")
df["nouns"] = df[BODY].fillna("").progress_apply(extract_nouns)

all_nouns = [w for lst in df["nouns"] for w in lst]
cnt = Counter(all_nouns)
print("총 토큰:", len(all_nouns), "| 고유:", len(cnt))
freq_df = pd.DataFrame(cnt.most_common(30), columns=["단어", "빈도"])
freq_df.head(15)

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
freq_df.set_index("단어")["빈도"][:20][::-1].plot.barh(ax=axes[0])
axes[0].set_title("단어 빈도 Top 20")

wc = WordCloud(font_path=NANUM, width=900, height=600,
               background_color="white", max_words=200
               ).generate_from_frequencies(dict(cnt.most_common(200)))
axes[1].imshow(wc, interpolation="bilinear"); axes[1].axis("off")
axes[1].set_title("워드클라우드")
plt.tight_layout(); plt.show()

**관찰**: 행정 상투어를 제거하면 `복지·가구·취약·계층·독거노인·고독사·센터·예방·안전·어르신·이웃·고립·사회보장·위험`이 부각된다.
'복지/지원'의 시혜 어휘와 '고립/위험/예방'의 위기 어휘가 공존한다.

## 3. LDA 토픽 모델

`filter_extremes(no_above=0.3)`로 과빈출어를 걷어내 토픽이 뭉개지는 것을 막는다.
토픽 수는 coherence로 탐색하되, c_v가 trivial한 작은 k를 과대평가하므로 4부터 탐색한다.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

token_lists = df["nouns"].tolist()
dictionary = Dictionary(token_lists)
dictionary.filter_extremes(no_below=5, no_above=0.3)
corpus = [dictionary.doc2bow(t) for t in token_lists]
print("사전 크기:", len(dictionary), "| 문서:", len(corpus))

rows = []
for k in range(4, 13, 2):
    m = LdaModel(corpus=corpus, id2word=dictionary, num_topics=k,
                 passes=5, random_state=42)
    cv = CoherenceModel(model=m, texts=token_lists, dictionary=dictionary,
                        coherence="c_v").get_coherence()
    rows.append((k, cv)); print(f"k={k}: coherence={cv:.4f}")
coh = pd.DataFrame(rows, columns=["k", "coherence"])
BEST_K = int(coh.loc[coh["coherence"].idxmax(), "k"])
print("선택된 토픽 수 k =", BEST_K)

In [ ]:
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=BEST_K,
               passes=15, iterations=100, random_state=42)

# 토픽 라벨 (해석) — 결과에 맞게 수정 가능
TOPIC_LABELS = {0: "봉사·나눔", 1: "노인의료·돌봄", 2: "재난안전대응",
                3: "취약계층복지", 4: "지역복지인프라", 5: "고독사예방"}
for tid in range(lda.num_topics):
    words = ", ".join(w for w, _ in lda.show_topic(tid, topn=10))
    print(f"[토픽 {tid}] {TOPIC_LABELS.get(tid, tid)}: {words}")

In [ ]:
# 문서별 대표 토픽
def dominant(bow):
    d = lda.get_document_topics(bow)
    return max(d, key=lambda x: x[1])[0] if d else -1
df["topic"] = [dominant(b) for b in corpus]
df["topic_label"] = df["topic"].map(TOPIC_LABELS)

dist = df["topic_label"].value_counts()
ax = dist.plot.bar(figsize=(9, 4), title="토픽별 기사 수")
ax.set_ylabel("기사 수"); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()
(dist / len(df) * 100).round(1)

In [ ]:
# pyLDAvis (인터랙티브)
import pyLDAvis, pyLDAvis.gensim_models as gv
pyLDAvis.enable_notebook()
gv.prepare(lda, corpus, dictionary)

## 4. 단어 동시출현 네트워크

빈도 상위 50개 단어를, 같은 기사에 300회 이상 함께 등장한 쌍으로 연결한다.

In [ ]:
import networkx as nx
G = build_network(token_lists, top_n=50, min_cooc=300)
print("노드:", G.number_of_nodes(), "| 엣지:", G.number_of_edges())

pos = nx.spring_layout(G, k=0.6, seed=42, weight="weight")
freqs = [G.nodes[n]["freq"] for n in G.nodes]; fmax = max(freqs)
sizes = [3000 * (f / fmax) + 200 for f in freqs]
ws = [G[u][v]["weight"] for u, v in G.edges]; wmax = max(ws)
widths = [3 * (w / wmax) + 0.2 for w in ws]

plt.figure(figsize=(14, 11))
nx.draw_networkx_edges(G, pos, width=widths, alpha=0.3)
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color="#4C9BE8", alpha=0.85)
nx.draw_networkx_labels(G, pos, font_size=11, font_family="NanumGothic")
plt.axis("off"); plt.title("단어 동시출현 네트워크", fontsize=14); plt.tight_layout(); plt.show()

centrality(G).head(10)

## 5. 감성 분석

KNU 한국어 감성사전(단어형)으로 기사별 점수를 매긴다. 명사만으로는 감성어를 못 잡으므로
`Okt.pos(stem=True)`로 형용사·동사 어간까지 복원해 매칭한다(약 1~2분).

In [ ]:
lexicon = load_lexicon(LEXICON_PATH)
print("감성사전 단어 수:", len(lexicon))

res = df[BODY].fillna("").progress_apply(lambda t: score_document(t, lexicon))
df[["senti_score", "senti_pos", "senti_neg", "senti_n", "senti_label"]] = \
    pd.DataFrame(res.tolist(), index=df.index)

dist = df["senti_label"].value_counts().reindex(["positive", "neutral", "negative"])
print(dist.to_string()); print("평균 점수:", round(df["senti_score"].mean(), 3))

colors = ["#4C9BE8", "#CCCCCC", "#E8734C"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dist.plot.bar(ax=axes[0], color=colors); axes[0].set_title("감성 라벨 분포"); axes[0].tick_params(axis="x", rotation=0)
df["senti_score"].clip(-15, 15).hist(bins=40, ax=axes[1], color="#4C9BE8"); axes[1].set_title("감성 점수 분포")
plt.tight_layout(); plt.show()

In [ ]:
# 월별 감성 추이
g = df.set_index("날짜").groupby([pd.Grouper(freq="ME"), "senti_label"]).size().unstack(fill_value=0)
score_m = df.set_index("날짜").resample("ME")["senti_score"].mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
score_m.plot(marker="o", ax=axes[0], color="#4C9BE8"); axes[0].axhline(0, color="gray", ls="--", lw=.8)
axes[0].set_title("월별 평균 감성 점수"); axes[0].grid(alpha=.3)
(g.div(g.sum(axis=1), axis=0))[["positive", "neutral", "negative"]].plot.area(
    ax=axes[1], color=colors, alpha=.8); axes[1].set_title("월별 감성 비중")
plt.tight_layout(); plt.show()

## 6. 토픽 × 감성 교차 — 모순의 근거

주제별로 감성이 어떻게 다른지 보면 '위험을 말할 때만 부정적'인 구조가 드러난다.

In [ ]:
cross = df.groupby("topic_label").agg(
    기사수=("senti_label", "size"),
    평균감성=("senti_score", "mean"),
    부정비율=("senti_label", lambda s: round((s == "negative").mean() * 100, 1)),
).sort_values("평균감성")
cross["평균감성"] = cross["평균감성"].round(2)
cross["비중(%)"] = (cross["기사수"] / len(df) * 100).round(1)
cross

## 7. 결론 — 모순된 현황

**고독사·독거노인은 명백한 사회적 '위험' 이슈지만, 언론은 이를 주로 복지·나눔이라는 '시혜적·긍정적' 프레임으로 다룬다.**
데이터는 이 모순을 일관되게 보여준다.

1. **프레임의 불균형** — 담론은 6개 주제로 갈리는데, 시혜·복지 프레임(봉사·나눔 + 취약계층복지 + 지역복지인프라 ≈ **50%**)이
   위기 프레임(고독사예방 + 재난안전대응 ≈ **38%**)보다 우세하다.
2. **전반적 긍정 편향** — 죽음·고립을 다루는 주제임에도 전체 감성은 긍정 약 61% / 부정 약 23%. '문제 고발'보다 '활동 홍보'가 많다.
3. **위험을 말할 때만 부정적** — 토픽×감성 교차에서 **재난안전대응·고독사예방만 부정 비율이 높고(약 27~30%)**,
   봉사·나눔·지역인프라는 강한 긍정이다. 즉 부정 정서는 실제 위험을 직접 다루는 좁은 영역에 갇혀 있다.
4. **'복지 대상'으로의 수렴** — 네트워크 중심어가 `복지·가구·독거노인·취약·계층`으로, 독거노인이 주로 *복지 수혜 대상*으로 호명된다.

### 함의
시혜·미담 중심의 보도는 사회적 관심을 환기하지만, **고독사·고립의 구조적 위험을 '개별 미담'으로 희석**할 수 있다.
복지 활동 홍보를 넘어 위험의 규모·원인·정책 공백을 다루는 보도가 상대적으로 적다는 점이 이 분석의 핵심 시사점이다.

### 분석의 한계
- 감성 점수는 **사전 기반**이라 문맥 부정("외롭지 **않다**")·반어를 반영하지 못한다 → 절대값보다 **주제 간·월별 상대 비교**로 해석.
- 긍정 편향은 실제 보도 성향일 수도, 사전 특성(봉사·지원 어휘가 긍정)일 수도 있어 단정은 피한다.
- 빅카인즈 키워드 OR 검색 기반이라 표본은 해당 키워드 보도에 한정된다.